In [6]:
import numpy as np 
import pandas as pd 
import os 
import glob
import warnings
warnings.filterwarnings('ignore')    ## I don't like pandas setting with copy warnings 
from scipy.stats import wilcoxon


In [3]:
def process_csvs(input_folder, output_folder):
    """
    Reads all CSVs from input_folder, removes rows where iso3 or Country == 'EMR',
    and saves them to output_folder with the same filename.
    """
    # Create output folder if it doesn't exist
    os.makedirs(output_folder, exist_ok=True)
    
    # Get all CSV files in input folder
    csv_files = glob.glob(os.path.join(input_folder, '*.csv'))
    
    for csv_file in csv_files:
        # Read the CSV
        df = pd.read_csv(csv_file)
        
        # Remove rows where iso3 or Country equals 'EMR'
        if 'iso3' in df.columns:
            df = df[df['iso3'] != 'EMR']
        if 'Country' in df.columns:
            df = df[df['Country'] != 'EMR']
        
        # Get filename and save to output folder
        filename = os.path.basename(csv_file)
        output_path = os.path.join(output_folder, filename)
        df.to_csv(output_path, index=False)
        print(f"Processed and saved: {filename}")

# Usage
process_csvs(input_folder='your_input_folder', output_folder='your_output_folder')

#in emissions v3 i have removed 1990 IRQ because of the incorrect unprocessed red meat data for it which was much higher than expected (174 g/d)... but there are EMR means included in the csvs which i should remove now
process_csvs(input_folder=r'co2_calc\emissions_v3', output_folder='co2_calc\emissions_v5')

Processed and saved: Beans_and_legumes_all.csv
Processed and saved: Beans_and_legumes_females.csv
Processed and saved: Beans_and_legumes_males.csv
Processed and saved: Cheese_all.csv
Processed and saved: Cheese_females.csv
Processed and saved: Cheese_males.csv
Processed and saved: Eggs_all.csv
Processed and saved: Eggs_females.csv
Processed and saved: Eggs_males.csv
Processed and saved: Fruits_all.csv
Processed and saved: Fruits_females.csv
Processed and saved: Fruits_males.csv
Processed and saved: Fruit_juices_all.csv
Processed and saved: Fruit_juices_females.csv
Processed and saved: Fruit_juices_males.csv
Processed and saved: Milk_all.csv
Processed and saved: Milk_females.csv
Processed and saved: Milk_males.csv
Processed and saved: Non_starchy_vegetables_all.csv
Processed and saved: Non_starchy_vegetables_females.csv
Processed and saved: Non_starchy_vegetables_males.csv
Processed and saved: Nuts_all.csv
Processed and saved: Nuts_females.csv
Processed and saved: Nuts_males.csv
Process

In [8]:

def test_regional_vs_global(
    regional_path: str, 
    global_path: str, 
    value_col: str = 'emissions', 
    time_col: str = 'year', 
    years_to_drop: list = None
):
    """
    Reads regional and global time-series data, drops specific years (e.g., pandemic anomalies),
    calculates the regional median per year, and runs a Wilcoxon signed-rank test.
    """
    reg_df = pd.read_csv(regional_path)
    # reg_df = reg_df[reg_df['Country'] != 'EMR'] 
    glob_df = pd.read_csv(global_path)

    if years_to_drop:
        reg_df = reg_df[~reg_df[time_col].isin(years_to_drop)]
        glob_df = glob_df[~glob_df[time_col].isin(years_to_drop)]

    reg_med = reg_df.groupby(time_col).agg({value_col: 'mean'}).reset_index()
    reg_med = reg_med.rename(columns={value_col: 'regional_mean'})

    glob_clean = glob_df[[time_col, value_col]].rename(columns={value_col: 'global_value'})

    merged_df = pd.merge(reg_med, glob_clean, on=time_col, how='inner')

    if merged_df.empty:
        raise ValueError("Merged dataframe is empty. Check column names and time overlaps.")

    stat, p_value = wilcoxon(merged_df['regional_mean'], merged_df['global_value'])
    
    print(f"--- Results for {value_col.upper()} ---")
    print(f"Years analyzed: {len(merged_df)} paired timepoints")
    print(f"Wilcoxon statistic: {stat:.4f}, p-value: {p_value:.4f}\n")

    return stat, p_value, merged_df


In [12]:
stat, p_val, paired_data = test_regional_vs_global(
    regional_path=r'co2_calc\emissions_v5\total_all.csv',
    global_path=r'co2_calc\global_emissions\total_all.csv',
    years_to_drop=[2020]
)
paired_data.round(1)

--- Results for EMISSIONS ---
Years analyzed: 7 paired timepoints
Wilcoxon statistic: 0.0000, p-value: 0.0156



,year,regional_mean,global_value
0,1990,2038.1,1928.7
1,1995,2221.4,1947.0
2,2000,2282.9,2015.3
3,2005,2335.5,2117.6
4,2010,2455.3,2290.2
5,2015,2442.1,2280.9
6,2018,2369.9,2309.4


In [13]:
stat, p_val, paired_data = test_regional_vs_global(
    regional_path=r'co2_calc\emissions_v5\total_males.csv',
    global_path=r'co2_calc\global_emissions\total_males.csv',
    years_to_drop=[2020]
)
paired_data.round(1)

--- Results for EMISSIONS ---
Years analyzed: 7 paired timepoints
Wilcoxon statistic: 0.0000, p-value: 0.0156



,year,regional_mean,global_value
0,1990,2061.3,1938.8
1,1995,2246.2,1960.3
2,2000,2311.7,2030.5
3,2005,2364.2,2129.7
4,2010,2485.0,2302.7
5,2015,2467.7,2293.6
6,2018,2394.8,2322.8


In [14]:
stat, p_val, paired_data = test_regional_vs_global(
    regional_path=r'co2_calc\emissions_v5\total_females.csv',
    global_path=r'co2_calc\global_emissions\total_females.csv',
    years_to_drop=[2020]
)
paired_data.round(1)

--- Results for EMISSIONS ---
Years analyzed: 7 paired timepoints
Wilcoxon statistic: 0.0000, p-value: 0.0156



,year,regional_mean,global_value
0,1990,1998.9,1918.8
1,1995,2179.0,1933.6
2,2000,2238.8,1998.7
3,2005,2291.4,2103.0
4,2010,2406.4,2276.6
5,2015,2395.1,2265.8
6,2018,2323.5,2294.3


In [20]:
import pandas as pd
from scipy.stats import wilcoxon

def run_comparison_co2(
    csv1_path: str, 
    csv2_path: str, 
    time_col: str, 
    country_col: str,
    drop_years: list = None,
    value_col: str = 'emissions' # Defaulting to your column name
):
    # 1. Load data
    df1 = pd.read_csv(csv1_path)
    df2 = pd.read_csv(csv2_path)

    # 2. Drop years if specified
    if drop_years:
        df1 = df1[~df1[time_col].isin(drop_years)]
        df2 = df2[~df2[time_col].isin(drop_years)]

    # 3. Merge on both time and country columns
    merged = pd.merge(
        df1, 
        df2, 
        on=[time_col, country_col], 
        how='inner', 
        suffixes=('_emr_males', '_emr_females')
    )
    col1, col2 = f'{value_col}_emr_males', f'{value_col}_emr_females'

    # 4. Safety check
    if merged.empty:
        raise ValueError("Merged dataframe is empty. Check your CSVs for overlapping times and countries.")

    # 5. Drop any rows where the target values might be NaN after merge (optional but recommended for Wilcoxon)
    merged = merged.dropna(subset=[col1, col2])

    # 6. Run the Wilcoxon paired test
    stat, p_val = wilcoxon(merged[col1], merged[col2])
    
    # 7. Output
    print("--- TEST SETTING: EMR VS EMR ---")
    print(f"Paired data points analyzed: {len(merged)}")
    print(f"Wilcoxon statistic: {stat:.4f}, p-value: {p_val:.4f}\n")

    return stat, p_val, merged

In [23]:
stat, p, data = run_comparison_co2(
    csv1_path=r'co2_calc\emissions_v5\total_males.csv',
    csv2_path=r'co2_calc\emissions_v5\total_females.csv',
    time_col='year',
    country_col='Country',
    drop_years=[2020]
)

data.round(1)

--- TEST SETTING: EMR VS EMR ---
Paired data points analyzed: 146
Wilcoxon statistic: 1586.0000, p-value: 0.0000



,Country,year,emissions_emr_males,emissions_emr_females
0,AFG,1990,1871.9,1871.1
1,AFG,1995,1606.6,1585.5
2,AFG,2000,1543.9,1512.3
3,AFG,2005,1431.5,1404.5
4,AFG,2010,1389.8,1359.4
...,...,...,...,...
141,YEM,2000,1144.6,1147.7
142,YEM,2005,1445.9,1441.5
143,YEM,2010,1425.5,1412.1
144,YEM,2015,1633.3,1614.8
